# TP2 — ChromaTruth (FFHQ + Inpainting)

"
"Este notebook reúne **todo o pipeline** do projeto (dados → geração de fakes → treino do colorizer → deteção/heatmaps → AUC/threshold).

"
"**Como usar (ordem recomendada):**
"
"1. Confirmar `config.py` (paths e parâmetros).
"
"2. `make_splits.py` (cria `real_train/real_test` + `splits.json`).
"
"3. `generate_fakes_inpainting.py` (gera fakes em `fake_test/` e máscaras em `masks/`).
"
"4. `train_colorizer.py` (treina U-Net small em Lab usando apenas reais).
"
"5. `build_pairs.py` (gera `pairs.csv`).
"
"6. `detect_and_xai.py` (gera heatmaps + `scores.csv`).
"
"7. `test_auc_coluna.py` (AUC por score). 
"
"8. `evaluate_threshold.py` (calibração + avaliação com threshold).

"
"Notas:
"
"- O código foi mantido igual ao que já está funcional; aqui só está **organizado** em células.
"
"- Para correr, basta executar as células pela ordem (ou correr os scripts via `!python ...`).
"


## 0) Setup do ambiente

In [ ]:
import os, sys
from pathlib import Path

# (opcional) garantir que o projeto está no PYTHONPATH
PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("PROJECT_DIR:", PROJECT_DIR)


## 0 config

Conteúdo original do ficheiro `config.py`:

In [ ]:
from pathlib import Path


# Nome da experiência
EXPERIMENT_NAME = "exp_inpainting_v1"


# Raiz do projeto
PROJECT_ROOT = Path(__file__).resolve().parent

# Pasta de dados
DATA_DIR = PROJECT_ROOT / "data"

# Pastas (Fase 1)
REAL_ALL_DIR = DATA_DIR / "real_all"
REAL_TRAIN_DIR = DATA_DIR / "real_train"
REAL_TEST_DIR = DATA_DIR / "real_test"
FAKE_TEST_DIR = DATA_DIR / "fake_test"

# Ficheiros
PAIRS_CSV = DATA_DIR / "pairs.csv"
SPLITS_JSON = DATA_DIR / "splits.json"

# Extensões permitidas
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

# Tamanhos
IMG_SIZE = 256  # manterf 256 por enquanto


## 1 logger utils

Conteúdo original do ficheiro `logger_utils.py`:

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

from config import (
    PROJECT_ROOT,
    EXPERIMENT_NAME,
)

# pasta base dos outputs
OUTPUTS_DIR = PROJECT_ROOT / "outputs" / "experiments" / EXPERIMENT_NAME
LOGS_DIR = OUTPUTS_DIR / "logs"


def ensure_dirs():
    LOGS_DIR.mkdir(parents=True, exist_ok=True)


class TeeLogger:
    def __init__(self, filepath: Path):
        self.file = open(filepath, "a", encoding="utf-8")
        self.stdout = sys.stdout
        self.stderr = sys.stderr

    def write(self, message):
        self.stdout.write(message)
        self.file.write(message)

    def flush(self):
        self.stdout.flush()
        self.file.flush()


def setup_logging(script_name: str):
    ensure_dirs()
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = LOGS_DIR / f"{script_name}_{ts}.log"

    logger = TeeLogger(log_file)
    sys.stdout = logger
    sys.stderr = logger

    print(f"[LOG] Experiência: {EXPERIMENT_NAME}")
    print(f"[LOG] A gravar logs em: {log_file}")


## 2 make splits

Conteúdo original do ficheiro `make_splits.py`:

In [ ]:
import json
import random
import shutil
from pathlib import Path

from config import REAL_ALL_DIR, REAL_TRAIN_DIR, REAL_TEST_DIR, IMG_EXTS, SPLITS_JSON
from logger_utils import setup_logging



def list_images(folder: Path):
    files = [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS]
    files.sort()
    return files


def safe_mkdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)


def main():
    setup_logging("make_splits")

    seed = 42

    train_ratio = 0.8
    test_ratio = 0.2

    assert abs(train_ratio + test_ratio - 1.0) < 1e-6, "As percentagens têm de somar 1.0"

    safe_mkdir(REAL_TRAIN_DIR)
    safe_mkdir(REAL_TEST_DIR)

    images = list_images(REAL_ALL_DIR)
    n_total = len(images)

    if n_total == 0:
        raise ValueError(f"Não há imagens em {REAL_ALL_DIR}")

    n_train = int(n_total * train_ratio)
    n_test = n_total - n_train

    random.seed(seed)
    random.shuffle(images)

    train_imgs = images[:n_train]
    test_imgs = images[n_train:]

    for p in train_imgs:
        shutil.copy2(p, REAL_TRAIN_DIR / p.name)

    for p in test_imgs:
        shutil.copy2(p, REAL_TEST_DIR / p.name)

    splits = {
        "seed": seed,
        "train_ratio": train_ratio,
        "test_ratio": test_ratio,
        "n_total": n_total,
        "n_train": n_train,
        "n_test": n_test,
        "train_files": [p.name for p in train_imgs],
        "test_files": [p.name for p in test_imgs],
    }

    with open(SPLITS_JSON, "w", encoding="utf-8") as f:
        json.dump(splits, f, indent=2)

    print(f"[OK] Total: {n_total} imagens")
    print(f"[OK] Train: {n_train} imagens ({train_ratio*100:.0f}%) -> {REAL_TRAIN_DIR}")
    print(f"[OK] Test:  {n_test} imagens ({test_ratio*100:.0f}%) -> {REAL_TEST_DIR}")



if __name__ == "__main__":
    main()


## 3 generate fakes inpainting

Conteúdo original do ficheiro `generate_fakes_inpainting.py`:

In [ ]:
import os
import time

# --- HOTFIX WINDOWS ---
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["MKL_THREADING_LAYER"] = "GNU"
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"

import random
from pathlib import Path

import torch
from PIL import Image, ImageDraw

from config import REAL_TEST_DIR, FAKE_TEST_DIR, IMG_EXTS
from logger_utils import setup_logging


def list_images(folder: Path):
    files = [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS]
    files.sort()
    return files


def make_random_mask(w: int, h: int, min_frac=0.18, max_frac=0.35) -> Image.Image: # tinha 0.25 0.55
    """
    Máscara branca (255) = região a inpaint
    Máscara preta (0) = manter
    Fazzer 1-3 retângulos aleatórios.
    """
    mask = Image.new("L", (w, h), 0)
    draw = ImageDraw.Draw(mask)

    # n = random.randint(1, 3)
    n = 1
    for _ in range(n):
        rw = int(random.uniform(min_frac, max_frac) * w)
        rh = int(random.uniform(min_frac, max_frac) * h)


        # x0 = random.randint(0, max(0, w - rw))
        # y0 = random.randint(0, max(0, h - rh))
        x_min = int(0.15 * w)
        x_max = int(0.85 * w) - rw
        y_min = int(0.15 * h)
        y_max = int(0.85 * h) - rh

        x0 = random.randint(x_min, max(x_min, x_max))
        y0 = random.randint(y_min, max(y_min, y_max))

        x1 = x0 + rw
        y1 = y0 + rh

        draw.rectangle([x0, y0, x1, y1], fill=255)

    return mask


def main():

    start_time = time.time()


    setup_logging("generate_fakes_inpainting")

    # --- 1) Modelo de inpainting (Diffusers) ---

    model_id = os.environ.get("INPAINT_MODEL", "runwayml/stable-diffusion-inpainting")

    prompt = os.environ.get("INPAINT_PROMPT", "a photo of a person, realistic, natural skin")
    negative_prompt = os.environ.get("INPAINT_NEG_PROMPT", "cartoon, anime, painting, deformed, distorted, low quality, blurry, extra fingers")

    # num_inference_steps = int(os.environ.get("INPAINT_STEPS", "8")) #quantos steps por imagem
    # guidance_scale = float(os.environ.get("INPAINT_GUIDANCE", "6.5")) #Forca que o modelo segue o prompt textual, valores baixos o modelo ignora o texto, mais altos forca demaisado e pode criar artefactos
    # strength = float(os.environ.get("INPAINT_STRENGTH", "0.75")) # 0 = imagem quase igual a original, 1  = imagem quase totalemnte regenerada
    seed = int(os.environ.get("INPAINT_SEED", "123"))

    NUM_INFERENCE_STEPS = 12
    GUIDANCE_SCALE = 5.5
    STRENGTH = 0.65
    MAX_IMAGES = 12000  # número de imagens a processar tinha 12000


    real_files = list_images(REAL_TEST_DIR)

    # Limitar número de imagens para não demorar tanto
    real_files = real_files[:MAX_IMAGES]

    print(f"[INFO] A gerar fakes para {len(real_files)} imagens")

    if not real_files:
        raise ValueError(f"Não encontrei imagens em {REAL_TEST_DIR}")

    FAKE_TEST_DIR.mkdir(parents=True, exist_ok=True)

    MASKS_DIR = FAKE_TEST_DIR.parent / "masks"
    MASKS_DIR.mkdir(parents=True, exist_ok=True)

    from diffusers import StableDiffusionInpaintPipeline

    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(seed)
    random.seed(seed)

    if device == "cuda":
        torch.cuda.manual_seed_all(seed)

    pipe = StableDiffusionInpaintPipeline.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    )
    pipe = pipe.to(device)

    pipe.safety_checker = None


    print(f"[INFO] Real test: {REAL_TEST_DIR}")
    print(f"[INFO] Fake test: {FAKE_TEST_DIR}")
    print(f"[INFO] Model: {model_id}")
    print(f"[INFO] Device: {device}")
    print(f"[INFO] Prompt: {prompt}")

    print(
        f"[INFO] Steps: {NUM_INFERENCE_STEPS} | "
        f"Guidance: {GUIDANCE_SCALE} | "
        f"Strength: {STRENGTH} | "
        f"MAX_IMAGES: {MAX_IMAGES}"
    )

    for i, real_path in enumerate(real_files, start=1):
        out_path = FAKE_TEST_DIR / real_path.name

        # Se já existe, salta
        if out_path.exists():
            continue

        img = Image.open(real_path).convert("RGB")
        w, h = img.size
        mask = make_random_mask(w, h)

        mask_out_path = MASKS_DIR / f"{real_path.stem}_mask.png"
        mask.save(mask_out_path)

        if device == "cuda":
            with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
                result = pipe(
                    prompt=prompt,
                    image=img,
                    mask_image=mask,
                    negative_prompt=negative_prompt,
                    num_inference_steps=NUM_INFERENCE_STEPS,
                    guidance_scale=GUIDANCE_SCALE,
                    strength=STRENGTH,
                ).images[0]
        else:
            with torch.no_grad():
                result = pipe(
                    prompt=prompt,
                    image=img,
                    mask_image=mask,
                    negative_prompt=negative_prompt,
                    num_inference_steps=NUM_INFERENCE_STEPS,
                    guidance_scale=GUIDANCE_SCALE,
                    strength=STRENGTH,
                ).images[0]

        result.save(out_path)
        if i % 100 == 0:
            print(f"#############################[OK] {i}/{len(real_files)} gerados#############################")

    print(f"[DONE] Gerados fakes para {len(real_files)} reais.")

    end_time = time.time()
    total_seconds = end_time - start_time

    print(f"[TIME] Tempo total de execução: {total_seconds / 60:.2f} minutos")


if __name__ == "__main__":
    main()


## 4 train colorizer

Conteúdo original do ficheiro `train_colorizer.py`:

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from logger_utils import setup_logging
from torch.cuda.amp import autocast, GradScaler

# -------------------------
# Configs rápidas
# -------------------------
PROJECT_DIR = Path(__file__).resolve().parent
DATA_DIR = PROJECT_DIR / "data"
REAL_TRAIN_DIR = DATA_DIR / "real_train"
REAL_VAL_DIR = DATA_DIR / "real_test"

OUTPUT_DIR = PROJECT_DIR / "outputs" / "colorizer"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ACC_THRESHOLD = 0.1

IMG_SIZE = 256
BATCH_SIZE = 32
EPOCHS = 50
LR = 1e-3
NUM_WORKERS = 4
PERSISTENT_WORKERS = True
PIN_MEMORY = True
PREFETCH_FACTOR = 2
MAX_TRAIN_IMAGES = 17000
MAX_VAL_IMAGES = 9000


# -------------------------
# Utils LAB
# -------------------------
def read_rgb(path: Path, size: int = 256):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)  # BGR
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    return img


def rgb_to_lab(rgb_uint8: np.ndarray):
    # OpenCV espera RGB em uint8 OK
    lab = cv2.cvtColor(rgb_uint8, cv2.COLOR_RGB2LAB).astype(np.float32)
    # L em [0,255], a/b em [0,255] (OpenCV shift)
    # Normalizar para treino estável:
    L = lab[..., 0:1] / 255.0                     # [0,1]
    ab = (lab[..., 1:3] - 128.0) / 128.0          # ~[-1,1]
    return L, ab


def list_images(folder: Path):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    files = [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in exts]
    files.sort()
    return files




# -------------------------
# Dataset
# -------------------------
class RealFacesLAB(Dataset):
    def __init__(self, folder: Path, max_images: int | None = None, img_size: int = 256):
        self.files = list_images(folder)
        if max_images is not None:
            self.files = self.files[:max_images]
        self.img_size = img_size

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        rgb = read_rgb(path, self.img_size)
        if rgb is None:
            # fallback: devolve outro
            return self.__getitem__((idx + 1) % len(self.files))

        L, ab = rgb_to_lab(rgb)

        # (H,W,C) -> (C,H,W)
        L = torch.from_numpy(L).permute(2, 0, 1).contiguous()     # (1,H,W)
        ab = torch.from_numpy(ab).permute(2, 0, 1).contiguous()   # (2,H,W)
        return L, ab


# -------------------------
# Modelo (U-Net simples e leve)
# Input: 1 canal (L)
# Output: 2 canais (ab)
# -------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNetSmall(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = ConvBlock(1, 32)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(32, 64)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(64, 128)
        self.pool3 = nn.MaxPool2d(2)

        self.mid = ConvBlock(128, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = ConvBlock(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = ConvBlock(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = ConvBlock(64, 32)

        self.out = nn.Conv2d(32, 2, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))

        m = self.mid(self.pool3(e3))

        d3 = self.up3(m)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return torch.tanh(self.out(d1))  # ab ~ [-1,1]



def run_epoch(model, loader, optimizer, criterion, device, train: bool, scaler=None, use_amp: bool = False):
    n = 0
    model.train(train)
    total_loss = 0.0
    total_pixels = 0
    correct_pixels = 0

    for i, (L, ab) in enumerate(loader):
        if i == 0:
            print("[DEBUG] primeira batch chegou")

        L = L.to(device, non_blocking=True)
        ab = ab.to(device, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=use_amp):
            pred = model(L)
            loss = criterion(pred, ab)

        if train:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        bs = L.size(0)
        total_loss += loss.item() * bs
        n += bs

        error = torch.abs(pred - ab)     # (B,2,H,W)
        error = torch.mean(error, dim=1) # (B,H,W)
        correct_pixels += (error < ACC_THRESHOLD).sum().item()
        total_pixels += error.numel()

    avg_loss = total_loss / max(n, 1)
    acc = correct_pixels / max(total_pixels, 1)
    return avg_loss, acc




def main():
    print("[DEBUG] A iniciar treino (primeira batch pode ser mais lenta)...")

    start = time.time()
    print("[DEBUG] antes do setup_logging")
    setup_logging("train_colorizer")
    print("[DEBUG] 1) a criar datasets...")

    if not REAL_TRAIN_DIR.exists():
        raise FileNotFoundError(f"Não existe: {REAL_TRAIN_DIR}")
    if not REAL_VAL_DIR.exists():
        raise FileNotFoundError(f"Não existe: {REAL_VAL_DIR}")

    device = "cuda" if torch.cuda.is_available() else "cpu"

    torch.backends.cudnn.benchmark = True

    use_amp = (device == "cuda")
    scaler = GradScaler(enabled=use_amp)

    model = UNetSmall().to(device)

    criterion = nn.SmoothL1Loss(beta=0.05)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2,
        threshold=1e-4,
        min_lr=1e-5,
        verbose=True,
    )

    print(f"[INFO] Device: {device}")
    print(f"[INFO] Train dir: {REAL_TRAIN_DIR}")
    print(f"[INFO] Val dir:   {REAL_VAL_DIR}")

    train_ds = RealFacesLAB(REAL_TRAIN_DIR, max_images=MAX_TRAIN_IMAGES, img_size=IMG_SIZE)
    val_ds = RealFacesLAB(REAL_VAL_DIR, max_images=MAX_VAL_IMAGES, img_size=IMG_SIZE)

    print(f"[DEBUG] 2 train_ds size = {len(train_ds)}")
    print(f"[DEBUG] 3 val_ds size   = {len(val_ds)}")

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=(PERSISTENT_WORKERS and NUM_WORKERS > 0),
        prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=(PERSISTENT_WORKERS and NUM_WORKERS > 0),
        prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None,
    )

    print("[DEBUG] 4) dataloaders criados")

    # model = UNetSmall().to(device)
    # criterion = nn.L1Loss()  # no PDF é permitido L1 ou MSE :contentReference[oaicite:1]{index=1}
    # optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    best_val = float("inf")
    best_path = OUTPUT_DIR / "best_colorizer.pt"



    # -------------------------
    # Early Stopping
    # -------------------------
    PATIENCE = 10
    MIN_DELTA = 1e-4
    epochs_no_improve = 0

    print("[DEBUG] 5) a iniciar loop de treino")

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = run_epoch(
            model, train_loader, optimizer, criterion, device,
            train=True, scaler=scaler, use_amp=use_amp
        )

        va_loss, va_acc = run_epoch(
            model, val_loader, optimizer, criterion, device,
            train=False, scaler=scaler, use_amp=use_amp
        )



        print(
            f"[EPOCH {epoch}/{EPOCHS}] "
            f"train_loss={tr_loss:.5f} | train_acc={tr_acc * 100:.2f}% || "
            f"val_loss={va_loss:.5f} | val_acc={va_acc * 100:.2f}%"
        )

        scheduler.step(va_loss)

        # Melhorou?
        if best_val - va_loss > MIN_DELTA:
            best_val = va_loss

            epochs_no_improve = 0

            torch.save(
                {
                    "model_state": model.state_dict(),
                    "epoch": epoch,
                    "val_loss": best_val,
                    "img_size": IMG_SIZE,
                },
                best_path,
            )
            print(f"[OK] Novo melhor modelo guardado: {best_path} (val_loss={best_val:.5f})")

        else:
            epochs_no_improve += 1
            print(f"[INFO] EarlyStopping: {epochs_no_improve}/{PATIENCE} sem melhoria")

            if epochs_no_improve >= PATIENCE:
                print("[STOP] Early stopping ativado")
                break

    elapsed = (time.time() - start) / 60.0
    print(f"[DONE] Treino concluído.")
    print(f"[TIME] Tempo total: {elapsed:.2f} minutos")


if __name__ == "__main__":
    main()


## 5 build pairs

Conteúdo original do ficheiro `build_pairs.py`:

In [ ]:
import csv
from pathlib import Path

from config import REAL_TEST_DIR, FAKE_TEST_DIR, PAIRS_CSV, IMG_EXTS
from logger_utils import setup_logging


def list_images(folder: Path):
    files = [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS]
    files.sort()
    return files


def main():
    setup_logging("build_pairs")


    real_files = list_images(REAL_TEST_DIR)

    # Limitar aos mesmos fakes gerados
    MAX_IMAGES = 6000
    real_files = real_files[:MAX_IMAGES]

    print(f"[INFO] A criar pares para {len(real_files)} imagens")

    if not real_files:
        raise ValueError(f"Não encontrei imagens em {REAL_TEST_DIR}.")

    missing = []
    rows = []

    for r in real_files:
        f = FAKE_TEST_DIR / r.name
        if not f.exists():
            missing.append(r.name)
            continue
        rows.append((str(r), str(f)))

    if missing:
        print("[ERRO] Estão a faltar fakes para estes ficheiros (mesmo nome!):")
        for name in missing[:30]:
            print(" -", name)
        if len(missing) > 30:
            print(f" ... e mais {len(missing) - 30}")
        raise ValueError("Gera os fakes em data/fake_test com o MESMO nome dos reais em data/real_test.")

    with open(PAIRS_CSV, "w", newline="", encoding="utf-8") as fp:
        w = csv.writer(fp)
        w.writerow(["real_path", "fake_path"])
        w.writerows(rows)

    print(f"[OK] pairs.csv criado com {len(rows)} pares -> {PAIRS_CSV}")


if __name__ == "__main__":
    main()


## 6 detect and xai

Conteúdo original do ficheiro `detect_and_xai.py`:

In [ ]:
import os

# --- HOTFIX WINDOWS (OpenMP) ---
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["MKL_THREADING_LAYER"] = "GNU"


import csv
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from config import PAIRS_CSV, PROJECT_ROOT
from logger_utils import setup_logging
from train_colorizer import UNetSmall, IMG_SIZE, read_rgb, rgb_to_lab


# --- Paths/outputs ---
MODEL_PATH = PROJECT_ROOT / "outputs" / "colorizer" / "best_colorizer.pt"
OUT_DIR = PROJECT_ROOT / "outputs" / "detection"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MASKS_DIR = PROJECT_ROOT / "data" / "masks"


def load_mask(mask_path: Path, target_shape_hw):
    # Lê máscara (0..255) e garante que tem o mesmo tamanho de E (H,W)
    m = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if m is None:
        raise ValueError(f"Falha a ler máscara: {mask_path}")

    H, W = target_shape_hw
    if m.shape[0] != H or m.shape[1] != W:
        m = cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST)

    return m


def read_pairs(csv_path: Path):
    rows = []
    with open(csv_path, "r", encoding="utf-8") as f:
        r = csv.DictReader(f)
        for row in r:
            rows.append((Path(row["real_path"]), Path(row["fake_path"])))
    return rows


def load_lab(path: Path):
    rgb = read_rgb(path, IMG_SIZE) # mesma leitura + resize do treino
    if rgb is None:
        raise ValueError(f"Falha ao ler imagem: {path}")

    L, ab = rgb_to_lab(rgb)  # mesma normalização do treino
    return L, ab



@torch.no_grad()
def predict_ab(model, L01: np.ndarray):
    x = torch.from_numpy(L01.transpose(2, 0, 1)).unsqueeze(0).float().to(DEVICE)

    if DEVICE == "cuda":
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            pred = model(x)
    else:
        pred = model(x)

    pred = pred.squeeze(0).cpu().numpy().transpose(1, 2, 0)
    return pred



def error_map(pred_ab: np.ndarray, orig_ab: np.ndarray):
    # E = ||ab_hat - ab_orig||_2 por pixel
    diff = pred_ab - orig_ab
    E = np.sqrt(np.sum(diff * diff, axis=2))  # (H,W)
    return E


def save_heatmap(E: np.ndarray,
                 out_path: Path,
                 rgb_uint8: np.ndarray = None,
                 top_percentile: float = 92.0,
                 dilate_iters: int = 2,
                 keep_largest: bool = True,
                 draw_bbox: bool = True):
    hm = E.astype(np.float32)
    hm = np.maximum(hm, 0.0)

    # Contraste robusto
    p_low, p_high = np.percentile(hm, [2.0, 99.5])
    hm = np.clip(hm, p_low, p_high)
    hm = (hm - hm.min()) / (hm.max() - hm.min() + 1e-8)

    # Suavização (menos "pontinhos")
    hm = cv2.GaussianBlur(hm, (0, 0), sigmaX=2.2, sigmaY=2.2)

    # Threshold menos agressivo para não ficar esparso
    thr = np.percentile(hm, top_percentile)
    mask = (hm >= thr).astype(np.uint8)  # 0/1

    # Dilatar para formar "mancha"
    if dilate_iters > 0:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
        mask = cv2.dilate(mask, kernel, iterations=dilate_iters)

    # Opcional: ficar só com a maior componente (uma zona principal)
    bbox = None
    if keep_largest:
        num, labels_cc, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
        if num > 1:
            best_j = None
            best_score = -1e18

            # ignora fundo (idx=0)
            for j in range(1, num):
                x, y, w, h, area = stats[j]
                if area < 60:
                    continue  # ignora manchas minúsculas

                comp = (labels_cc == j)

                # energia = soma das intensidades do heatmap dentro da componente
                energy = float(hm[comp].sum())

                # bónus leve por área (estabilidade, mas não domina)
                score = energy + 0.001 * float(area)

                if score > best_score:
                    best_score = score
                    best_j = j

            if best_j is not None:
                mask = (labels_cc == best_j).astype(np.uint8)
                x, y, w, h, area = stats[best_j]
                bbox = (x, y, w, h)

    # Aplicar máscara no heatmap para ficar limpo
    hm = hm * mask.astype(np.float32)

    # Color map
    hm_u8 = (hm * 255).astype(np.uint8)
    hm_color = cv2.applyColorMap(hm_u8, cv2.COLORMAP_JET)

    if rgb_uint8 is None:
        cv2.imwrite(str(out_path), hm_color)
        return

    img_bgr = cv2.cvtColor(rgb_uint8, cv2.COLOR_RGB2BGR)
    overlay = cv2.addWeighted(img_bgr, 0.60, hm_color, 0.40, 0)

    # Desenhar bounding box da principal região
    if draw_bbox and bbox is not None:
        x, y, w, h = bbox
        cv2.rectangle(overlay, (x, y), (x + w, y + h), (0, 255, 0), 2)

    cv2.imwrite(str(out_path), overlay)


def diff_ab_map(ab_real: np.ndarray, ab_fake: np.ndarray):
    # delta = ||ab_fake - ab_real||_2 por pixel
    diff = ab_fake - ab_real
    D = np.sqrt(np.sum(diff * diff, axis=2))  # (H,W)
    return D


def roc_auc(scores, labels):
    # Implementação simples (sem sklearn) para não depender de nada.
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int32)

    order = np.argsort(-scores)  # desc
    labels = labels[order]

    P = np.sum(labels == 1)
    N = np.sum(labels == 0)
    if P == 0 or N == 0:
        return None, None, None

    tps = np.cumsum(labels == 1)
    fps = np.cumsum(labels == 0)
    tpr = tps / P
    fpr = fps / N

    # AUC trapezoidal
    auc = np.trapezoid(tpr, fpr)
    return fpr, tpr, auc


def main():

    # Para decidir automaticamente se o score tem de ser invertido
    comp_real_list = []
    comp_fake_list = []

    setup_logging("detect_and_xai")

    if not MODEL_PATH.exists():
        raise FileNotFoundError(f"Modelo não encontrado: {MODEL_PATH}")

    pairs = read_pairs(PAIRS_CSV)
    print(f"[INFO] PARES: {len(pairs)} -> {PAIRS_CSV}")

    model = UNetSmall().to(DEVICE)
    ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    print(f"[INFO] Modelo carregado: {MODEL_PATH} | DEVICE={DEVICE}")

    rows_out = []
    scores = []
    labels = []

    # Para não explodir disco, só guardo heatmaps de alguns exemplos
    heatmaps_dir = OUT_DIR / "heatmaps"
    heatmaps_dir.mkdir(parents=True, exist_ok=True)

    for i, (real_p, fake_p) in enumerate(pairs, start=1):
        # --- REAL ---
        Lr, abr = load_lab(real_p)
        pred_r = predict_ab(model, Lr)
        Er = error_map(pred_r, abr)

        #METER EM COMENTARIO PARA JA DEPOIS VE-SE SE DESCOMENTO OU NAO
        mask_path_real = MASKS_DIR / f"{real_p.stem}_mask.png"
        mask_path_fake = MASKS_DIR / f"{fake_p.stem}_mask.png"

        mask_path = None
        if mask_path_fake.exists():
            mask_path = mask_path_fake
        elif mask_path_real.exists():
            mask_path = mask_path_real

        if mask_path is not None:
            m = load_mask(mask_path, Er.shape)
            masked_pixels = (m > 0)
        else:
            masked_pixels = None

        # --- FAKE ---
        Lf, abf = load_lab(fake_p)
        pred_f = predict_ab(model, Lf)
        Ef = error_map(pred_f, abf)

        # =========================
        # NOVOS SCORES (melhor recall)
        # =========================
        eps = 1e-8

        # -------------------------
        # Vetores sem máscara (global)
        # -------------------------
        Er_all = Er.reshape(-1)
        Ef_all = Ef.reshape(-1)

        # -------------------------
        # Vetores com máscara (ROI)
        # Se a máscara for pequena, cai para global para não instabilizar.
        # -------------------------
        min_pixels = 100

        if masked_pixels is None:
            Er_roi = Er_all
            Ef_roi = Ef_all
        else:
            Er_roi = Er[masked_pixels]
            Ef_roi = Ef[masked_pixels]
            if Er_roi.size < min_pixels or Ef_roi.size < min_pixels:
                Er_roi = Er_all
                Ef_roi = Ef_all

        # ---------- GLOBAL ----------
        P_HIGH = 85 #melhor -> 85

        p99_r = float(np.percentile(Er_all, P_HIGH))
        p99_f = float(np.percentile(Ef_all, P_HIGH))

        p95_r = float(np.percentile(Er_all, 95))
        p95_f = float(np.percentile(Ef_all, 95))

        k_all = max(1, int(0.01 * Er_all.size))
        top1_r = float(np.mean(np.partition(Er_all, -k_all)[-k_all:]))
        top1_f = float(np.mean(np.partition(Ef_all, -k_all)[-k_all:]))

        # ---------- MASK/ROI ----------
        p99_r_m = float(np.percentile(Er_roi, P_HIGH))
        p99_f_m = float(np.percentile(Ef_roi, P_HIGH))

        p95_r_m = float(np.percentile(Er_roi, 95))
        p95_f_m = float(np.percentile(Ef_roi, 95))

        k_roi = max(1, int(0.01 * Er_roi.size))
        top1_r_m = float(np.mean(np.partition(Er_roi, -k_roi)[-k_roi:]))
        top1_f_m = float(np.mean(np.partition(Ef_roi, -k_roi)[-k_roi:]))


        # -------------------------
        # SCORES POR IMAGEM (não constantes)
        # -------------------------
        score_p99_real = p99_r
        score_p99_fake = p99_f

        score_top1_real = top1_r
        score_top1_fake = top1_f

        # Score composto simples (melhora ranking/AUC)
        score_comp_real = score_p99_real + score_top1_real
        score_comp_fake = score_p99_fake + score_top1_fake

        comp_real_list.append(score_comp_real)
        comp_fake_list.append(score_comp_fake)


        # -------------------------
        # SCORES POR PAR (mantemos para análise)
        # -------------------------
        score_ratio_p99 = (p99_f + eps) / (p99_r + eps)
        score_delta_top1 = top1_f - top1_r


        score_ratio_p99_mask = (p99_f_m + eps) / (p99_r_m + eps)
        score_delta_top1_mask = top1_f_m - top1_r_m

        score_log_ratio_p99 = float(np.log(score_ratio_p99 + eps))
        score_log_ratio_p99_mask = float(np.log(score_ratio_p99_mask + eps))

        # --- NOVO 1: PEAK RATIO (mais estável que p99.99 sozinho) ---
        peak_f = (p99_f - p95_f)
        peak_r = (p99_r - p95_r)
        score_peak_ratio = float((peak_f + eps) / (peak_r + eps))

        peak_f_m = (p99_f_m - p95_f_m)
        peak_r_m = (p99_r_m - p95_r_m)
        score_peak_ratio_mask = float((peak_f_m + eps) / (peak_r_m + eps))

        # --- NOVO 2: SCORE MIX (combina log_ratio + delta_top1) ---
        LAMBDA_DELTA = 0.9 # melhor com 0.9
        score_mix = float(score_log_ratio_p99 + LAMBDA_DELTA * score_delta_top1)
        score_mix_mask = float(score_log_ratio_p99_mask + LAMBDA_DELTA * score_delta_top1_mask)


        # -------------------------
        # FUSÃO DE SCORES (por par) depois posso tirar
        # -------------------------
        # Normalização simples por par:
        # - ratio_p99 já é "normalizado" (divide pela dificuldade do real)
        # - delta_top1 pode ser positivo/negativo, damos-lhe um peso menor
        alpha = 1.0 #melhor com 1.5
        fused_score = score_ratio_p99 + alpha * score_delta_top1


        rows_out.append([str(real_p), 0,
                         p99_r, top1_r, (p99_r + top1_r),
                         1.0, 0.0,
                         1.0, 0.0,
                         fused_score,
                         0.0, 0.0,
                         1.0, 1.0,  # peak_ratio (real = 1.0 para manter formato)
                         0.0, 0.0])  # score_mix (real = 0.0)

        rows_out.append([str(fake_p), 1,
                         p99_f, top1_f, (p99_f + top1_f),
                         score_ratio_p99, score_delta_top1,
                         score_ratio_p99_mask, score_delta_top1_mask,
                         fused_score,
                         score_log_ratio_p99, score_log_ratio_p99_mask,
                         score_peak_ratio, score_peak_ratio_mask,
                         score_mix, score_mix_mask])




        # Para ROC/AUC rápida neste script, usamos o score_comp
        # scores.append(score_comp_real); labels.append(0)
        # scores.append(score_comp_fake); labels.append(1) # Ao fazer com este dava 0.52

        # scores.append(1.0); labels.append(0)
        # scores.append(score_ratio_p99); labels.append(1) # Ao fazer com este dava 0.65

        # ESTE FOI O QUE OBTIVE SEGUNDO MELHOR RESULTADO
        # scores.append(1.0); labels.append(0)
        # scores.append(fused_score); labels.append(1)

        # scores.append(1.0); labels.append(0)
        # scores.append(score_ratio_tail); labels.append(1)

        #Sem mascara 0.7574
        # scores.append(0.0); labels.append(0)
        # scores.append(score_log_ratio_p99); labels.append(1)

        # Sem mascara (usar score_mix)Melhor resultado! 0.7627
        scores.append(0.0); labels.append(0)
        scores.append(score_mix); labels.append(1)

        # Guardar heatmaps só para alguns exemplos
        if i in {1, 10, 100, 500, 1000}:
            # heatmap que localiza a alteração (diferença direta real vs fake em ab)
            D = diff_ab_map(abr, abf)

            rgb_real = read_rgb(real_p, IMG_SIZE)
            rgb_fake = read_rgb(fake_p, IMG_SIZE)

            save_heatmap(D, heatmaps_dir / f"diffab_fake_{i:05d}.png", rgb_fake, top_percentile=90.0, dilate_iters=2,keep_largest=True, draw_bbox=True)
            save_heatmap(D, heatmaps_dir / f"diffab_real_{i:05d}.png", rgb_real, top_percentile=90.0, dilate_iters=2,keep_largest=True, draw_bbox=True)

            # Os erros do modelo (Er/Ef) podes deixar mais suaves também
            save_heatmap(Er, heatmaps_dir / f"realerr_{i:05d}.png", rgb_real, top_percentile=90.0, dilate_iters=1,keep_largest=True, draw_bbox=False)
            save_heatmap(Ef, heatmaps_dir / f"fakeerr_{i:05d}.png", rgb_fake, top_percentile=90.0, dilate_iters=1,keep_largest=True, draw_bbox=False)

        if i % 200 == 0:
            print(f"[OK] {i}/{len(pairs)} pares processados")

    # =========================
    # Ajustar orientação do score_comp (muito importante!)
    # Quero: fakes com score MAIOR que reais.
    # =========================
    mean_real = float(np.mean(comp_real_list)) if len(comp_real_list) > 0 else 0.0
    mean_fake = float(np.mean(comp_fake_list)) if len(comp_fake_list) > 0 else 0.0
    print(f"[DEBUG] mean(score_comp) real={mean_real:.6f} | fake={mean_fake:.6f}")

    # Guardar scores
    scores_csv = OUT_DIR / "scores.csv"
    with open(scores_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(
            ["path", "label",
             "score_p99", "score_top1_mean", "score_comp",
             "score_ratio_p99", "score_delta_top1_mean",
             "score_ratio_p99_mask", "score_delta_top1_mean_mask",
             "fused_score",
             "score_log_ratio_p99", "score_log_ratio_p99_mask",
             "score_peak_ratio", "score_peak_ratio_mask",
             "score_mix", "score_mix_mask"]
        )

        w.writerows(rows_out)

    # ROC/AUC
    fpr, tpr, auc = roc_auc(scores, labels)
    if auc is None:
        print("[WARN] Não consegui calcular ROC/AUC (labels insuficientes).")
    else:
        print(f"[RESULT] AUC = {auc:.4f}")

        plt.figure()
        plt.plot(fpr, tpr)
        plt.xlabel("FPR")
        plt.ylabel("TPR")
        plt.title(f"ROC (AUC={auc:.4f})")
        plt.tight_layout()
        plt.savefig(OUT_DIR / "roc_curve.png", dpi=200)
        plt.close()

    print(f"[DONE] Guardado: {scores_csv}")
    print(f"[DONE] Guardado: {OUT_DIR / 'roc_curve.png'} (se AUC calculada)")
    print(f"[DONE] Heatmaps: {heatmaps_dir}")


if __name__ == "__main__":
    main()


## 7 test auc coluna

Conteúdo original do ficheiro `test_auc_coluna.py`:

In [ ]:
import csv
import numpy as np
from pathlib import Path

SCORES_CSV = Path("outputs/detection/scores.csv")

def roc_auc(scores, labels):
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int32)

    order = np.argsort(-scores)
    labels = labels[order]

    P = np.sum(labels == 1)
    N = np.sum(labels == 0)
    if P == 0 or N == 0:
        return None

    tps = np.cumsum(labels == 1)
    fps = np.cumsum(labels == 0)
    tpr = tps / P
    fpr = fps / N
    auc = np.trapezoid(tpr, fpr)
    return float(auc)

def load_col(col):
    y, s = [], []
    with open(SCORES_CSV, "r", encoding="utf-8") as f:
        r = csv.DictReader(f)
        for row in r:
            y.append(int(row["label"]))
            s.append(float(row[col]))
    return np.array(y), np.array(s)

def main():
    cols = ["score_p99", "score_top1_mean", "score_comp", "score_ratio_p99", "score_delta_top1_mean"]
    for c in cols:
        y, s = load_col(c)
        auc = roc_auc(s, y)
        print(f"[AUC] {c}: {auc:.4f}")

if __name__ == "__main__":
    main()

## 8 evaluate threshold

Conteúdo original do ficheiro `evaluate_threshold.py`:

In [ ]:
import csv
from pathlib import Path
import numpy as np

SCORES_CSV = Path("outputs/detection/scores.csv")
SCORE_COL = "score_mix"

def load_scores(path: Path):
    y = []
    s = []
    with open(path, "r", encoding="utf-8") as f:
        r = csv.DictReader(f)
        for row in r:
            y.append(int(row["label"]))
            s.append(float(row[SCORE_COL]))
    return np.array(y, dtype=np.int32), np.array(s, dtype=np.float64)

def confusion(y_true, y_pred):
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    return tp, tn, fp, fn


def roc_auc(scores, labels):
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int32)

    order = np.argsort(-scores)  # desc
    labels = labels[order]

    P = np.sum(labels == 1)
    N = np.sum(labels == 0)
    if P == 0 or N == 0:
        return None

    tps = np.cumsum(labels == 1)
    fps = np.cumsum(labels == 0)
    tpr = tps / P
    fpr = fps / N

    auc = np.trapezoid(tpr, fpr)
    return float(auc)


def metrics(tp, tn, fp, fn):
    acc = (tp + tn) / max(tp + tn + fp + fn, 1)
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-12)

    beta = 2.0
    f2 = (1 + beta**2) * prec * rec / max((beta**2) * prec + rec, 1e-12)
    return acc, prec, rec, f1, f2


def best_threshold(y, s):
    thresh = np.unique(s)
    best = None

    for t in thresh:
        y_pred = (s >= t).astype(np.int32)
        tp, tn, fp, fn = confusion(y, y_pred)
        acc, prec, rec, f1, f2 = metrics(tp, tn, fp, fn)
        score = f1

        cand = {"t": float(t), "score": float(score), "tp": tp, "tn": tn, "fp": fp, "fn": fn,
                "acc": acc, "prec": prec, "rec": rec, "f1": f1, "f2": f2}

        if best is None:
            best = cand
        else:
            # 1) melhor score (F1 ou F2)
            if cand["score"] > best["score"] + 1e-12:
                best = cand
            # 2) empate no score: preferir MENOS FP
            elif abs(cand["score"] - best["score"]) <= 1e-12 and cand["fp"] < best["fp"]:
                best = cand
            # 3) novo empate: preferir threshold mais alto (mais conservador)
            elif abs(cand["score"] - best["score"]) <= 1e-12 and cand["fp"] == best["fp"] and cand["t"] > best["t"]:
                best = cand

    return best

def main():
    y, s = load_scores(SCORES_CSV)

    auc_all = roc_auc(s, y)
    if auc_all is not None:
        print(f"[AUC] Global ({SCORE_COL}): {auc_all:.4f}")

    # -------------------------------------------------
    # Split por "grupo/par": mantém real+fake juntos
    # Grupo = stem do filename do campo "path" no CSV
    # -------------------------------------------------
    paths = []
    with open(SCORES_CSV, "r", encoding="utf-8") as f:
        r = csv.DictReader(f)
        for row in r:
            paths.append(row["path"])

    groups = np.array([Path(p).stem for p in paths], dtype=object)

    uniq = np.unique(groups)

    rng = np.random.default_rng(123)
    rng.shuffle(uniq)

    cut = int(0.8 * len(uniq))
    g_cal = set(uniq[:cut])
    g_eval = set(uniq[cut:])

    idx_cal = np.array([i for i, g in enumerate(groups) if g in g_cal], dtype=np.int64)
    idx_eval = np.array([i for i, g in enumerate(groups) if g in g_eval], dtype=np.int64)

    print(f"[SPLIT] cal: {len(idx_cal)} | eval: {len(idx_eval)}")
    print(f"[SPLIT] cal positives={int(np.sum(y[idx_cal] == 1))} negatives={int(np.sum(y[idx_cal] == 0))}")
    print(f"[SPLIT] eval positives={int(np.sum(y[idx_eval] == 1))} negatives={int(np.sum(y[idx_eval] == 0))}")

    auc_eval = roc_auc(s[idx_eval], y[idx_eval])
    if auc_eval is not None:
        print(f"[AUC] Eval  ({SCORE_COL}): {auc_eval:.4f}")


    best = best_threshold(y[idx_cal], s[idx_cal])
    t = best["t"]



    y_pred_eval = (s[idx_eval] >= t).astype(np.int32)
    tp, tn, fp, fn = confusion(y[idx_eval], y_pred_eval)
    acc, prec, rec, f1, f2 = metrics(tp, tn, fp, fn)

    print(f"[CAL] best_threshold (F1): {t:.6f} | F1_cal={best['f1']:.4f} | F2_cal={best['f2']:.4f}")
    print("[EVAL] Confusion: TP={} TN={} FP={} FN={}".format(tp, tn, fp, fn))
    print("[EVAL] Acc={:.4f} Prec={:.4f} Rec={:.4f} F1={:.4f} F2={:.4f}".format(acc, prec, rec, f1, f2))


if __name__ == "__main__":
    main()


## 9) Executar o pipeline via comandos (opcional)


In [ ]:
# Ajusta os nomes conforme o teu repositório (se estiverem noutra pasta).
# Exemplo:
# !python make_splits.py
# !python generate_fakes_inpainting.py
# !python train_colorizer.py
# !python build_pairs.py
# !python detect_and_xai.py
# !python test_auc_coluna.py
# !python evaluate_threshold.py
